In [12]:
from pathlib import Path
import contextlib
import hashlib
import importlib
import io
import json
import os
import re
import sys
from time import perf_counter

import numpy as np
import pandas as pd
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
import CoolProp.CoolProp as CP

# Set this explicitly if the notebook is moved outside the eBATS tree.
PROJECT_ROOT_OVERRIDE = None


def locate_project(override=None):
    starts = [Path(override)] if override else [Path.cwd(), Path('D:/eBATS')]
    for start in starts:
        for root in [start.resolve(), *start.resolve().parents]:
            if (root / 'lib/BTMS_model.py').is_file() and (root / 'data/C0010').is_dir():
                return root
    raise FileNotFoundError('Set PROJECT_ROOT_OVERRIDE to the eBATS project directory.')


PROJECT_ROOT = locate_project(PROJECT_ROOT_OVERRIDE)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
BTMS_model = importlib.reload(importlib.import_module('lib.BTMS_model'))

RESULT_BASENAME = 'C0010R02_PARK25_ACLC_INTG_1D_EVAL'
# Same result location and direct Excel export convention as C0008/C0009.
RESULT_DATA_DIR = PROJECT_ROOT / 'data' / 'C0010'
RESULT_EXCEL_PATH = Path(os.environ.get(
    'C0010_RESULT_EXCEL_PATH', RESULT_DATA_DIR / f'{RESULT_BASENAME}.xlsx'))
DT_S = 1.0  # Keep 1 s to match C0008/C0009 and the source reports.
AMBIENTS_C = (20, 25, 35)
T_LIMIT_C, DELTA_T_LIMIT_C = 45.0, 5.0
# Use the same public iterative solver and settings as C0008/C0009.
# Reuse validated C0008/C0009 results and completed C0010 cases; solve only missing cases.
SOLVER_SETTINGS = dict(solver_tol=1e-6, solver_maxiter=1000)
# Existing upstream-trajectory validation tolerances also apply to iterative results.
BALANCE_RESIDUAL_TOL_W = 1e-6
ENERGY_CLOSURE_TOL_J = 1e-3
CASE_WORKBOOK_DIR = RESULT_EXCEL_PATH.parent / f'{RESULT_BASENAME}_cases'
CACHE_DATA_DIR = Path(os.environ.get('C0010_CACHE_DIR', CASE_WORKBOOK_DIR))
FORCE_RECALCULATE = False
USE_UPSTREAM_RESULTS = True
REUSE_C0010_RESULTS = True
BATTERY = dict(mass_kg=0.048, cp_J_kgK=830.0, diameter_m=0.018, height_m=0.065)
CELL_VOLUME_M3 = np.pi / 4 * BATTERY['diameter_m']**2 * BATTERY['height_m']
AC = dict(velocity_m_s=1.5, gap_m=0.002, rows=16, columns=20,
          fan_efficiency=0.35, plate_thickness_m=0.002, plate_density_kg_m3=2700.0)
LC = dict(m_dot_total_kg_s=0.045, channels=32, segments=20, transverse_cells=16,
          width_m=0.010, height_m=0.003, length_m=0.380,
          min_spacing_m=0.001, bottom_m=0.002, k_plate_W_mK=200.0,
          rho_plate_kg_m3=2719.0, pump_efficiency=0.35)
MISSIONS = {
    'short': dict(distance_km=50, cruise_s=633, hover_s=300, duration_s=1293,
                  source=PROJECT_ROOT / 'data/C0010/1293.out', source_type='volumetric_out'),
    'baseline': dict(distance_km=100, cruise_s=1380, hover_s=180, duration_s=1920,
                     source=PROJECT_ROOT / 'data/MD05E070207A1  data_power gen_single cell_Liu_20260421.xlsx',
                     source_type='single_cell_excel'),
    'long': dict(distance_km=120, cruise_s=1680, hover_s=180, duration_s=2220,
                 source=PROJECT_ROOT / 'data/C0010/2220.out', source_type='volumetric_out'),
}
STAGE_NAMES = ('takeoff', 'climb', 'transition1', 'cruise',
               'transition2', 'descent', 'hover', 'landing')
CRATES = (3.6, 3.0, 1.5, 1.0, 0.36, 2.0, 4.0, 4.0)
for mission in MISSIONS.values():
    durations = [6, 30, 144, mission['cruise_s'], 144, 30, mission['hover_s'], 6]
    mission['bounds_s'] = np.r_[0, np.cumsum(durations)]
    if mission['bounds_s'][-1] != mission['duration_s']:
        raise ValueError('Mission stages do not sum to the mission duration.')
if DT_S != 1.0:
    raise ValueError('This revision uses DT_S = 1 s to reproduce C0008/C0009.')
print('Project:', PROJECT_ROOT)
print('Model-result workbook:', RESULT_EXCEL_PATH)

Project: D:\eBATS
Model-result workbook: D:\eBATS\data\C0010\C0010R02_PARK25_ACLC_INTG_1D_EVAL.xlsx


In [13]:
def read_heat(mission_key, mission):
    source = mission['source']
    duration = mission['duration_s']
    if not source.is_file():
        raise FileNotFoundError(source)
    duplicate_count = 0
    if mission['source_type'] == 'single_cell_excel':
        data = pd.read_excel(source, sheet_name='Sheet1')
        if 'Qbat(W)' not in data:
            raise ValueError(f'Missing Qbat(W): {source}')
        values = pd.to_numeric(data['Qbat(W)'], errors='raise')
        last = values.last_valid_index()
        if last is None or values.loc[:last].isna().any():
            raise ValueError('Baseline heat contains an empty or internal missing sample.')
        q = values.loc[:last].to_numpy(dtype=float)
        if len(q) not in (duration, duration + 1):
            raise ValueError(f'Baseline needs {duration} or {duration+1} valid samples, got {len(q)}.')
        raw = pd.DataFrame({'source_time_s': np.arange(len(q)), 'raw_value': q})
        raw_unit = 'W/cell'
    else:
        lines = source.read_text(encoding='utf-8-sig').splitlines()
        headers = re.findall(r'"([^"]+)"', lines[2])
        heat_column = 'report-def-0'
        if 'flow-time' not in headers or heat_column not in headers:
            raise ValueError(f'Unexpected heat report header in {source}: {headers}')
        data = pd.read_csv(source, sep=r'\s+', skiprows=3, names=headers, dtype=float)
        if data.empty or not np.isfinite(data.to_numpy()).all():
            raise ValueError(f'Empty or non-finite heat report: {source}')
        duplicate_count = int(data.duplicated('flow-time').sum())
        data = data.sort_values('flow-time', kind='stable').drop_duplicates('flow-time', keep='last')
        source_times = data['flow-time'].to_numpy()
        if len(data) != duration + 1 or not np.allclose(source_times, np.arange(duration+1), atol=1e-8, rtol=0):
            raise ValueError(f'Heat report must cover all 1-s samples from 0 to {duration}: {source}')
        raw = pd.DataFrame({'source_time_s': source_times, 'raw_value': data[heat_column].to_numpy()})
        q = raw['raw_value'].to_numpy() * CELL_VOLUME_M3
        raw_unit = 'W/m3'
    if not np.isfinite(q).all() or np.any(q < 0):
        raise ValueError(f'Invalid heat-generation values: {source}')
    raw.insert(0, 'Mission_key', mission_key)
    raw['raw_unit'] = raw_unit
    raw['Q_cell_W'] = q
    raw['used_for_integration'] = raw['source_time_s'] < duration
    intervals = pd.DataFrame({'Mission_key': mission_key,
                              'interval_start_s': np.arange(duration),
                              'interval_end_s': np.arange(1, duration+1),
                              'Q_cell_W': q[:duration]})
    metadata = dict(Mission_key=mission_key, **{k: mission[k] for k in
                    ('distance_km', 'cruise_s', 'hover_s', 'duration_s', 'source_type')},
                    source_path=str(source), source_sha256=hashlib.sha256(source.read_bytes()).hexdigest(),
                    raw_unit=raw_unit, source_samples=len(raw), restart_duplicates_removed=duplicate_count,
                    cell_volume_m3=CELL_VOLUME_M3,
                    E_heat_cell_J=float(np.sum(q[:duration]) * DT_S),
                    Q_peak_cell_W=float(np.max(q[:duration])))
    return q[:duration].copy(), raw, intervals, metadata


heat_inputs, raw_heat_tables, interval_tables, source_rows = {}, [], [], []
for mission_key, mission in MISSIONS.items():
    heat_inputs[mission_key], raw, intervals, metadata = read_heat(mission_key, mission)
    raw_heat_tables.append(raw)
    interval_tables.append(intervals)
    source_rows.append(metadata)
source_table = pd.DataFrame(source_rows)
raw_heat_table = pd.concat(raw_heat_tables, ignore_index=True)
heat_interval_table = pd.concat(interval_tables, ignore_index=True)
print(source_table[['Mission_key', 'duration_s', 'raw_unit', 'Q_peak_cell_W', 'E_heat_cell_J']].to_string(index=False))

Mission_key  duration_s raw_unit  Q_peak_cell_W  E_heat_cell_J
      short        1293     W/m3       3.841393    1481.244985
   baseline        1920   W/cell       4.153141    1290.228597
       long        2220     W/m3       4.633758    1438.136909


In [14]:
def fluid_properties(fluid, ambient):
    result = {name: float(CP.PropsSI(code, 'T', ambient + 273.15, 'P', 101325.0, fluid))
              for name, code in [('rho', 'D'), ('mu', 'V'), ('cp', 'C'), ('k', 'L')]}
    result['Pr'] = result['cp'] * result['mu'] / result['k']
    return result


def design(cooling, ambient):
    props = fluid_properties('Air' if cooling == 'AC' else 'Water', ambient)
    if cooling == 'AC':
        segments, cells_per_node, paths = AC['columns'], 1.0, AC['rows']
        st = BATTERY['diameter_m'] + AC['gap_m']
        sl = np.sqrt(3) / 2 * st
        length = BATTERY['diameter_m'] + (AC['columns']-1) * sl
        width = BATTERY['diameter_m'] + (AC['rows']-1) * st
        area_cs = width / AC['rows'] * BATTERY['height_m']
        velocity = AC['velocity_m_s']
        re_number = props['rho'] * velocity * BATTERY['diameter_m'] / props['mu']
        eps = BTMS_model.select_correction_factor(AC['columns'], re_number)
        nu = BTMS_model.zukauskas_nusellet(re_number, props['Pr'], eps, props['Pr'])
        htc = nu * props['k'] / BATTERY['diameter_m']
        area_ht = np.pi * BATTERY['diameter_m'] * BATTERY['height_m']
        mass = AC['plate_density_kg_m3'] * length * width * AC['plate_thickness_m']
        pressure_drop, pump_power = 0.0, 0.0
    else:
        segments, paths = LC['segments'], LC['channels']
        cells_per_node = LC['transverse_cells'] / paths
        length = LC['length_m']
        width = max(LC['transverse_cells'] * BATTERY['diameter_m'],
                    paths * LC['width_m'] + (paths+1) * LC['min_spacing_m'])
        area_cs = LC['width_m'] * LC['height_m']
        diameter = 2 * LC['width_m'] * LC['height_m'] / (LC['width_m']+LC['height_m'])
        area_ht = (LC['width_m'] + 2 * LC['height_m']) * length / segments
        velocity = LC['m_dot_total_kg_s'] / paths / (props['rho'] * area_cs)
        re_number = props['rho'] * velocity * diameter / props['mu']
        nu = BTMS_model.liquid_nusselt_number_rect_channel(LC['width_m'], LC['height_m'])
        htc_fluid = nu * props['k'] / diameter
        htc = 1 / (1 / htc_fluid + LC['bottom_m'] / LC['k_plate_W_mK'])
        friction = BTMS_model.cal_friction_factor_rect_channel(re_number, LC['width_m'], LC['height_m'])
        pump = BTMS_model.cal_btms_aux_power(dict(
            fluid_cool='Water', rho_cool=props['rho'], mu_cool=props['mu'], A_cool_cs=area_cs,
            D_channel=diameter, L_channel=length, m_dot_total=LC['m_dot_total_kg_s'],
            num_channel=paths, pump_efficiency=LC['pump_efficiency'], K_minor=0.0, operation_time=1.0),
            friction_factor=friction)
        pump_power, pressure_drop = pump['P_aux_W'], pump['delta_p_Pa']
        mass = BTMS_model.cal_btms_mass(dict(fluid_cool='Water', rho_cool=props['rho'],
            rho_plate=LC['rho_plate_kg_m3'], L_channel=length, W_plate=width,
            A_cool_cs=area_cs, H_channel=LC['height_m'], H_bottom=LC['bottom_m'],
            num_channel=paths, m_pump=0.0, m_pipe=0.0))
    return dict(cooling=cooling, ambient=ambient, props=props, segments=segments,
                cells_per_node=cells_per_node, paths=paths, area_cs=area_cs,
                area_ht=area_ht, velocity=velocity, Re=re_number, Nu=nu, htc=htc,
                mass_kg=mass, length_m=length, width_m=width,
                pump_power_W=pump_power, pressure_drop_Pa=pressure_drop,
                C_node_J_K=BATTERY['mass_kg'] * BATTERY['cp_J_kgK'] * cells_per_node,
                G_node_W_K=htc * area_ht,
                F_path_W_K=props['rho'] * velocity * area_cs * props['cp'])


def flight_velocity(time_s, mission):
    b = mission['bounds_s']
    v190, v241 = 190/3.6, 241/3.6
    return np.interp(time_s, [b[0], b[1], b[2], b[3], b[4], b[5], b[6], b[-1]],
                     [0, 0, v190, v241, v241, v190, 0, 0])


def stage_index(time_s, mission):
    return np.minimum(np.searchsorted(mission['bounds_s'][1:], time_s, side='right'), 7)


designs = {(cooling, ambient): design(cooling, ambient)
           for cooling in ('AC', 'LC') for ambient in AMBIENTS_C}

In [15]:
def solver_step_args(d, q, previous, coolant_previous):
    return dict(num_seg=d['segments'], num_seg_bat=d['segments'], dt=DT_S,
                T_bat_pre=previous.copy(), T_cool_pre=coolant_previous.copy(),
                u_cool_in=d['velocity'], p_cool=101325.0,
                fluid_cool='Air' if d['cooling']=='AC' else 'Water',
                A_HT_seg=d['area_ht'], A_cool_cs=d['area_cs'],
                m_bat=BATTERY['mass_kg'], cp_bat=BATTERY['cp_J_kgK'],
                D_bat=BATTERY['diameter_m'], T_cool_in=d['ambient'],
                T_cool_out=max(float(coolant_previous[-1]), d['ambient']),
                is_cool=True, htc_cool=np.full(d['segments'], d['htc']),
                cp_cool=d['props']['cp'], rho_cool=d['props']['rho'],
                Q_gen=float(q), num_cell_seg=d['cells_per_node'], debug=False)


def run_case(cooling, mission_key, ambient):
    mission, d, q = MISSIONS[mission_key], designs[cooling, ambient], heat_inputs[mission_key]
    duration, nseg = mission['duration_s'], d['segments']
    time = np.arange(duration+1, dtype=float)
    bat = np.full((duration+1, nseg), float(ambient))
    coolant = bat.copy()
    v_x = flight_velocity(time[:-1], mission)
    power = np.full(duration, d['pump_power_W'])
    if cooling == 'AC':
        # Same reference fan inlet area as C0008; gap changes only the thermal geometry.
        inlet_area = (BATTERY['diameter_m'] + (AC['rows']-1)*1.25*BATTERY['diameter_m']) * BATTERY['height_m']
        power = np.array([BTMS_model.cal_air_fan_supplement_power(dict(
            rho_air=d['props']['rho'], mu_air=d['props']['mu'], V_air_in=AC['velocity_m_s'],
            V_x=v, D_bat=BATTERY['diameter_m'], N_c=AC['columns'], A_in=inlet_area,
            fan_efficiency=AC['fan_efficiency']))['P_fan_W'] for v in v_x])
    solver_seconds = 0.0
    for k in range(duration):
        args = solver_step_args(d, q[k], bat[k], coolant[k])
        solver_start = perf_counter()
        with contextlib.redirect_stdout(io.StringIO()):
            distribution = BTMS_model.solve_coolant_temperature_distribution(
                args, tol=SOLVER_SETTINGS['solver_tol'],
                maxiter=SOLVER_SETTINGS['solver_maxiter'], debug=False)
        solver_seconds += perf_counter() - solver_start
        distribution = np.asarray(distribution, dtype=float)
        if distribution.shape != (2*nseg,) or not np.isfinite(distribution).all():
            raise FloatingPointError(
                f'Invalid solver result: {cooling}/{mission_key}/{ambient}, step {k+1}')
        coolant[k+1] = distribution[:nseg]
        bat[k+1] = distribution[nseg:]
    result = assemble_case(cooling, mission_key, ambient, bat, coolant, power)
    result[2].update(Solver_time_s=solver_seconds, Solver_calls=duration,
                     Solver='BTMS_model.solve_coolant_temperature_distribution',
                     Solver_tol=SOLVER_SETTINGS['solver_tol'],
                     Solver_maxiter=SOLVER_SETTINGS['solver_maxiter'])
    return result



def assemble_case(cooling, mission_key, ambient, bat, coolant, power):
    # Only validate and summarize an existing trajectory; no transient solve here.
    mission, d, q = MISSIONS[mission_key], designs[cooling, ambient], heat_inputs[mission_key]
    duration, nseg = mission['duration_s'], d['segments']
    time = np.arange(duration+1, dtype=float)
    C, G, F = d['C_node_J_K'], d['G_node_W_K'], d['F_path_W_K']
    tin = np.column_stack([np.full(duration, ambient), coolant[1:, :-1]])
    ht = G * (bat[1:] - (tin + coolant[1:])/2)
    stored = C * np.diff(bat, axis=0) / DT_S
    r_bat = d['cells_per_node']*q[:, None] - stored - ht
    r_cool = ht - F*(coolant[1:]-tin)
    residual = float(max(np.max(np.abs(r_bat)), np.max(np.abs(r_cool))))
    if residual > BALANCE_RESIDUAL_TOL_W or not np.isfinite(bat).all() or not np.isfinite(coolant).all():
        raise AssertionError(f'Invalid temperatures/energy residual {residual}')
    upper, lower = bat.max(axis=1), bat.min(axis=1)
    difference = upper-lower
    peak, spread = int(upper.argmax()), int(difference.argmax())
    removed = F * (coolant[1:, -1]-ambient) * d['paths']
    cumulative_aux = np.r_[0.0, np.cumsum(power)*DT_S]
    module_cells = d['paths'] * nseg * d['cells_per_node']
    cumulative_generated = np.r_[0.0, np.cumsum(q)*DT_S*module_cells]
    cumulative_removed = np.r_[0.0, np.cumsum(removed)*DT_S]
    stored_module = d['paths']*C*np.sum(bat-ambient, axis=1)
    closure = float(np.max(np.abs(cumulative_generated-cumulative_removed-stored_module)))
    if closure > ENERGY_CLOSURE_TOL_J:
        raise AssertionError(f'Module energy does not close: {closure} J')
    case_id = f'{cooling}_{mission_key}_T{ambient:03d}C'
    history = pd.DataFrame(dict(time_s=time, Tmax_C=upper, Tmin_C=lower,
        Tmean_C=bat.mean(axis=1), DeltaT_C=difference, Tcool_out_C=coolant[:, -1],
        # Following-step quantities are null at the terminal state, not extended.
        Q_cell_next_interval_W=np.r_[q, np.nan], P_aux_next_interval_W=np.r_[power, np.nan],
        E_aux_cumulative_J=cumulative_aux, E_heat_module_cumulative_J=cumulative_generated,
        E_removed_module_cumulative_J=cumulative_removed, E_stored_module_J=stored_module,
        Stage=np.asarray(STAGE_NAMES)[stage_index(time, mission)]))
    history.insert(0, 'CaseID', case_id)
    history['Cooling'], history['Mission_key'], history['Ambient_C'] = cooling, mission_key, ambient
    summary = dict(CaseID=case_id, Cooling=cooling, Mission_key=mission_key, Ambient_C=ambient,
        Distance_km=mission['distance_km'], Duration_s=duration, dt_s=DT_S,
        Tmax_C=float(upper[peak]), time_of_Tmax_s=float(time[peak]),
        DeltaTmax_C=float(difference[spread]), time_of_DeltaTmax_s=float(time[spread]),
        final_Tmax_C=float(upper[-1]), final_Tmin_C=float(lower[-1]),
        Temperature_margin_C=T_LIMIT_C-float(upper[peak]),
        DeltaT_margin_C=DELTA_T_LIMIT_C-float(difference[spread]),
        Thermal_feasible=bool(upper[peak]<=T_LIMIT_C and difference[spread]<=DELTA_T_LIMIT_C),
        E_aux_J=float(cumulative_aux[-1]), Component_mass_kg=float(d['mass_kg']),
        E_heat_cell_J=float(np.sum(q)*DT_S), E_heat_module_J=float(cumulative_generated[-1]),
        Energy_closure_max_J=closure, Balance_residual_max_W=residual,
        Re=float(d['Re']), Nu=float(d['Nu']), htc_W_m2K=float(d['htc']),
        Number_of_segments=nseg, Cells_per_segment=d['cells_per_node'], Parallel_paths=d['paths'],
        Heat_source=str(mission['source']), cooling_mode='all_mission')
    detailed = pd.DataFrame(bat, columns=[f'T_bat_{i+1:02d}_C' for i in range(nseg)])
    for i in range(nseg):
        detailed[f'T_cool_{i+1:02d}_C'] = coolant[:, i]
    detailed.insert(0, 'time_s', time)
    stage_rows = []
    for index, name in enumerate(STAGE_NAMES):
        start, end = mission['bounds_s'][index:index+2].astype(int)
        # Include both boundary states; adjacent stages share their boundary state only.
        s = slice(start, end+1)
        stage_rows.append(dict(CaseID=case_id, Cooling=cooling, Mission_key=mission_key,
            Ambient_C=ambient, Stage=name, start_s=start, end_s=end,
            Tmax_C=float(upper[s].max()), DeltaTmax_C=float(difference[s].max()),
            end_Tmax_C=float(upper[end]), end_Tmin_C=float(lower[end]),
            end_Tmean_C=float(bat[end].mean()), E_aux_J=float(np.sum(power[start:end])*DT_S)))
    return history, detailed, summary, stage_rows, []

In [16]:
# Reuse completed cases first; only missing/invalid cases call the public iterative solver.
import types


def code_payload(code):
    def constant(value):
        if isinstance(value, types.CodeType):
            return code_payload(value)
        if isinstance(value, bytes):
            return ['bytes', value.hex()]
        if isinstance(value, tuple):
            return ['tuple', [constant(item) for item in value]]
        if value is None or isinstance(value, (str, bool, int, float)):
            return value
        return repr(value)
    return [code.co_code.hex(), list(code.co_names),
            [constant(value) for value in code.co_consts]]


def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


CALCULATION_CODE_SHA256 = hashlib.sha256(json.dumps(
    [code_payload(fn.__code__) for fn in (fluid_properties, design, flight_velocity,
                                        stage_index, solver_step_args, run_case, assemble_case)],
    sort_keys=True, separators=(',', ':')).encode()).hexdigest()
MODEL_SHA256 = sha256_file(BTMS_model.__file__)
HEAT_SHA256 = {key: sha256_file(mission['source']) for key, mission in MISSIONS.items()}
CACHE_SCHEMA = 2


def input_signature(cooling, mission_key, ambient):
    mission = MISSIONS[mission_key]
    payload = dict(schema=CACHE_SCHEMA, cooling=cooling, mission=mission_key,
        ambient=float(ambient), dt=DT_S, battery=BATTERY, solver_settings=SOLVER_SETTINGS,
        settings=AC if cooling == 'AC' else LC,
        mission_parameters={key: mission[key] for key in
                            ('distance_km', 'cruise_s', 'hover_s', 'duration_s')},
        stages=mission['bounds_s'].tolist(), heat_sha256=HEAT_SHA256[mission_key],
        model_sha256=MODEL_SHA256, calculation_sha256=CALCULATION_CODE_SHA256,
        coolprop_version=CP.get_global_param_string('version'))
    return hashlib.sha256(json.dumps(payload, sort_keys=True, separators=(',', ':')).encode()).hexdigest()



def auxiliary_power(cooling, mission_key, ambient):
    mission, d = MISSIONS[mission_key], designs[cooling, ambient]
    if cooling == 'LC':
        return np.full(mission['duration_s'], d['pump_power_W'])
    inlet_area = (BATTERY['diameter_m']+(AC['rows']-1)*1.25*BATTERY['diameter_m'])*BATTERY['height_m']
    velocity = flight_velocity(np.arange(mission['duration_s']), mission)
    return np.array([BTMS_model.cal_air_fan_supplement_power(dict(
        rho_air=d['props']['rho'], mu_air=d['props']['mu'], V_air_in=AC['velocity_m_s'],
        V_x=value, D_bat=BATTERY['diameter_m'], N_c=AC['columns'],
        A_in=inlet_area, fan_efficiency=AC['fan_efficiency']))['P_fan_W'] for value in velocity])


def expected_parameters(cooling, ambient):
    if cooling == 'AC':
        return dict(V_air=AC['velocity_m_s'], T_air=ambient, s_gap_mm=AC['gap_m']*1000)
    return dict(T_water_in_C=ambient, m_dot_total_kg_s=LC['m_dot_total_kg_s'],
        num_parallel_channels=LC['channels'], W_channel_mm=LC['width_m']*1000,
        H_channel_mm=LC['height_m']*1000, L_channel_mm=LC['length_m']*1000)


def read_validated_trajectory(path, selected_sheet, cooling, mission_key, ambient, cache=False):
    with pd.ExcelFile(path) as workbook:
        summary = pd.read_excel(workbook, sheet_name='Summary')
        if cache or len(summary) == 1:
            if len(summary) != 1:
                raise ValueError('Expected exactly one case in the Summary worksheet.')
            row = summary.iloc[0]
        else:
            temperature_column = 'Ambient_C' if 'Ambient_C' in summary else 'T_water_in_C'
            matched = summary.loc[np.isclose(summary[temperature_column], ambient)]
            if len(matched) != 1:
                raise ValueError('No unique matching upstream ambient condition.')
            row = matched.iloc[0]
        for column, value in expected_parameters(cooling, ambient).items():
            if column not in row or not np.isclose(float(row[column]), value, atol=1e-9, rtol=1e-9):
                raise ValueError(f'Upstream/cached parameter mismatch: {column}')
        if row.get('model') != 'Selected' or row.get('cooling_mode', 'all_mission') != 'all_mission':
            raise ValueError('Only the selected, all-mission cooling configuration can be reused.')
        duration = MISSIONS[mission_key]['duration_s']
        if not np.isclose(row['simulation_duration_s'], duration) or not np.isclose(row['dt_s'], DT_S):
            raise ValueError('Mission duration/time step mismatch.')
        if cache:
            if tuple(row.get(key) for key in ('Cooling', 'Mission_key', 'Ambient_C')) != (cooling, mission_key, ambient):
                raise ValueError('Cached case identity mismatch.')
            # Revalidate the complete trajectory against current physical inputs below.
            # A bookkeeping/solver-code fingerprint change alone does not force a new solve.
        selected = pd.read_excel(workbook, sheet_name=selected_sheet)
        if 'Sheet1' in workbook.sheet_names:
            nodes = pd.read_excel(workbook, sheet_name='Sheet1')
            time_column = next((key for key in ('time', 'time_s', 'Time Step') if key in nodes), None)
            battery_candidates = [[f'T_bat_{i+1}' for i in range(20)],
                                  [f'T_bat {i+1}' for i in range(20)]]
            coolant_candidates = [[f'T_cool_{i+1}' for i in range(20)],
                                  [f'T_air {i+1}' for i in range(20)]]
            battery_columns = next((cols for cols in battery_candidates if set(cols).issubset(nodes)), None)
            coolant_columns = next((cols for cols in coolant_candidates if set(cols).issubset(nodes)), None)
            if time_column is None or battery_columns is None or coolant_columns is None:
                raise ValueError('Detailed node columns are missing.')
            node_time = nodes[time_column].to_numpy(dtype=float)
            bat = nodes[battery_columns].to_numpy(dtype=float)
            coolant = nodes[coolant_columns].to_numpy(dtype=float)
        else:
            prefix = 'T_water' if cooling == 'LC' else 'T_air'
            battery_columns = [f'T_bat_seg_{i+1:02d}_C' for i in range(20)]
            coolant_columns = [f'{prefix}_seg_{i+1:02d}_C' for i in range(20)]
            bat = selected[battery_columns].to_numpy(dtype=float)
            coolant = selected[coolant_columns].to_numpy(dtype=float)
            node_time = selected['time_s'].to_numpy(dtype=float)
    expected_time = np.arange(duration+1, dtype=float)
    if (len(selected) != duration+1 or bat.shape != (duration+1, 20)
        or coolant.shape != bat.shape or len(node_time) != duration+1
        or not np.allclose(node_time, expected_time, atol=1e-8, rtol=0)
        or not np.allclose(selected['time_s'], expected_time, atol=1e-8, rtol=0)):
        raise ValueError('Missing, repeated or incomplete full-mission time records.')
    if not np.isfinite(bat).all() or not np.isfinite(coolant).all():
        raise ValueError('Invalid historical node temperatures.')
    if not np.allclose(np.r_[bat[0], coolant[0]], ambient, atol=1e-8, rtol=0):
        raise ValueError('Historical initial temperature mismatch.')
    for column, values in [('Tmax_C', bat.max(axis=1)), ('Tmin_C', bat.min(axis=1)),
                           ('DeltaT_C', np.ptp(bat, axis=1))]:
        if not np.allclose(selected[column], values, atol=1e-8, rtol=0):
            raise ValueError(f'Historical curves and nodes disagree: {column}')
    outlet = 'Twater_out_C' if cooling == 'LC' else 'Tair_out_C'
    if not np.allclose(selected[outlet], coolant[:,-1], atol=1e-8, rtol=0):
        raise ValueError('Historical outlet temperatures disagree.')
    power = auxiliary_power(cooling, mission_key, ambient)
    result = assemble_case(cooling, mission_key, ambient, bat, coolant, power)
    if cache:
        for column in ('Q_cell_next_interval_W', 'P_aux_next_interval_W',
                       'E_aux_cumulative_J', 'E_heat_module_cumulative_J',
                       'E_removed_module_cumulative_J', 'E_stored_module_J'):
            if not np.allclose(selected[column], result[0][column], atol=1e-6, rtol=1e-10, equal_nan=True):
                raise ValueError(f'Cached forcing/energy data changed: {column}')
    return result, row


def upstream_candidate(cooling, mission_key, ambient):
    if mission_key != 'baseline':
        return None
    if cooling == 'AC' and ambient == 25:
        return (PROJECT_ROOT/'data/C0008/C0008R01_PARK25_AC_1D_EVAL.xlsx', 'Selected')
    if cooling == 'LC' and ambient == 25:
        return (PROJECT_ROOT/'data/C0009/C0009R01_PARK25_LC_1D_EVAL.xlsx', 'Selected')
    if cooling == 'LC':
        return (PROJECT_ROOT/'data/C0009/C0009R01_PARK25_LC_1D_EVAL_AMBIENT.xlsx', f'Selected_{ambient:g}C')
    return None


def select_case(cooling, mission_key, ambient):
    case_id = f'{cooling}_{mission_key}_T{ambient:03d}C'
    rejection_notes = []
    load_started = perf_counter()
    candidate = upstream_candidate(cooling, mission_key, ambient)
    if not FORCE_RECALCULATE and USE_UPSTREAM_RESULTS and candidate is not None:
        path, sheet = candidate
        try:
            result, row = read_validated_trajectory(path, sheet, cooling, mission_key, ambient)
            result[2].update(
                Historical_calculation_time_s=row.get('Historical_calculation_time_s', row.get('Calculation_time_s', np.nan)),
                Historical_solver_time_s=row.get('Historical_solver_time_s', row.get('Solver_time_s', np.nan)),
                Calculation_time_s=0.0, Solver_time_s=0.0, Solver_calls=0,
                Result_load_time_s=perf_counter()-load_started)
            return result, dict(Action='upstream', Result_origin='upstream',
                Result_source=str(path), Source_sheet=sheet, Source_sha256=sha256_file(path),
                Reuse_note='Matched parameters, full time axis and current heat/energy balances.')
        except (OSError, ValueError, KeyError, AssertionError) as error:
            rejection_notes.append(f'Upstream not reusable: {error}')
    cached = CACHE_DATA_DIR/f'{RESULT_BASENAME}_{case_id}.xlsx'
    if not FORCE_RECALCULATE and REUSE_C0010_RESULTS and cached.is_file():
        try:
            result, row = read_validated_trajectory(cached, 'Selected', cooling, mission_key, ambient, cache=True)
            result[2].update(
                Historical_calculation_time_s=row.get('Historical_calculation_time_s', row.get('Calculation_time_s', np.nan)),
                Historical_solver_time_s=row.get('Historical_solver_time_s', row.get('Solver_time_s', np.nan)),
                Calculation_time_s=0.0, Solver_time_s=0.0, Solver_calls=0,
                Result_load_time_s=perf_counter()-load_started)
            origin = row.get('Result_origin', 'computed')
            return result, dict(Action='cache', Result_origin=origin,
                Result_source=str(row.get('Result_source', cached)),
                Source_sheet=str(row.get('Source_sheet', 'Selected')),
                Source_sha256=str(row.get('Source_sha256', sha256_file(cached))),
                Reuse_note='Validated current inputs/model and complete cached trajectory.')
        except (OSError, ValueError, KeyError, AssertionError) as error:
            rejection_notes.append(f'Cache not reusable: {error}')
    print(f'Computing {case_id}: no reusable result; public iterative solver', flush=True)
    lookup_seconds = perf_counter()-load_started
    started = perf_counter()
    result = run_case(cooling, mission_key, ambient)
    result[2].update(Calculation_time_s=perf_counter()-started, Result_load_time_s=lookup_seconds)
    return result, dict(Action='computed', Result_origin='computed',
        Result_source=str(MISSIONS[mission_key]['source']), Source_sheet='heat input',
        Source_sha256=HEAT_SHA256[mission_key],
        Reuse_note='; '.join(rejection_notes) if rejection_notes else
                   ('Forced recalculation.' if FORCE_RECALCULATE else 'No reusable case result.'))


histories, temperatures, summaries, stages, verification, reuse_records = {}, {}, [], [], [], []
for mission_key in MISSIONS:
    for ambient in AMBIENTS_C:
        for cooling in ('AC', 'LC'):
            result, provenance = select_case(cooling, mission_key, ambient)
            history, detailed, summary, stage_rows, legacy_rows = result
            case_id = summary['CaseID']
            summary.update(provenance)
            summary.update(Input_signature=input_signature(cooling, mission_key, ambient),
                Cache_schema=CACHE_SCHEMA, Input_heat_sha256=HEAT_SHA256[mission_key],
                Input_model_sha256=MODEL_SHA256, Calculation_code_sha256=CALCULATION_CODE_SHA256,
                Property_version=CP.get_global_param_string('version'))
            histories[case_id], temperatures[case_id] = history, detailed
            summaries.append(summary); stages.extend(stage_rows); verification.extend(legacy_rows)
            reuse_records.append(dict(CaseID=case_id, **provenance,
                Calculation_time_s=summary['Calculation_time_s'], Result_load_time_s=summary['Result_load_time_s'], Solver_time_s=summary['Solver_time_s'],
                Solver_calls=summary['Solver_calls']))
            print(f"{case_id}: {provenance['Action']}; new calculation={summary['Calculation_time_s']:.3f} s; "
                  f"Tmax={summary['Tmax_C']:.4f} C; DeltaTmax={summary['DeltaTmax_C']:.4f} C",
                  flush=True)
summary_table = pd.DataFrame(summaries)
stage_table = pd.DataFrame(stages)
verification_table = pd.DataFrame(verification, columns=[
    'Cooling','Mission_key','Ambient_C','time_s','max_abs_difference_C'])
reuse_table = pd.DataFrame(reuse_records)
if len(summary_table) != 18 or summary_table.duplicated(['Cooling','Mission_key','Ambient_C']).any():
    raise AssertionError('Expected exactly 18 unique cases.')
regression_table = pd.DataFrame(columns=['Cooling','Reference','Variable','max_abs_difference_C','Status'])
summary_table['model'] = 'Selected'
summary_table['source_notebook'] = 'C0010_PARK25_ACLC_INTG_1D_EVAL.ipynb'
print('Run actions:', reuse_table['Action'].value_counts().to_dict())
print(f"Total new case calculation time: {summary_table['Calculation_time_s'].sum():.3f} s")


Computing AC_short_T020C: no reusable result; public iterative solver
AC_short_T020C: computed; new calculation=27.461 s; Tmax=46.5163 C; DeltaTmax=5.0434 C
Computing LC_short_T020C: no reusable result; public iterative solver
LC_short_T020C: computed; new calculation=70.409 s; Tmax=32.8668 C; DeltaTmax=4.4585 C
Computing AC_short_T025C: no reusable result; public iterative solver
AC_short_T025C: computed; new calculation=25.955 s; Tmax=51.6024 C; DeltaTmax=5.0876 C
Computing LC_short_T025C: no reusable result; public iterative solver
LC_short_T025C: computed; new calculation=25.964 s; Tmax=37.8015 C; DeltaTmax=4.4918 C
Computing AC_short_T035C: no reusable result; public iterative solver
AC_short_T035C: computed; new calculation=25.849 s; Tmax=61.7707 C; DeltaTmax=5.1740 C
Computing LC_short_T035C: no reusable result; public iterative solver
LC_short_T035C: computed; new calculation=26.048 s; Tmax=47.6855 C; DeltaTmax=4.5465 C
Computing AC_baseline_T020C: no reusable result; public it

In [17]:
# Each case follows C0009's Summary / Selected / Sheet1 interface.
# The index workbook stores one row per case and points to its separate workbook.
CASE_WORKBOOK_DIR = RESULT_EXCEL_PATH.parent / f'{RESULT_BASENAME}_cases'

heat_generation_table = raw_heat_table.rename(columns={'source_time_s': 'time_s'}).copy()
heat_generation_table['interval_end_s'] = np.where(
    heat_generation_table['used_for_integration'], heat_generation_table['time_s'] + DT_S, np.nan)

configuration = dict(
    result_id=RESULT_BASENAME, dt_s=DT_S,
    model_file=str(Path(BTMS_model.__file__).resolve()),
    model_sha256=hashlib.sha256(Path(BTMS_model.__file__).read_bytes()).hexdigest(),
    coolprop_version=CP.get_global_param_string('version'),
    battery=BATTERY, cell_volume_m3=CELL_VOLUME_M3, AC=AC, LC=LC,
    T_limit_C=T_LIMIT_C, DeltaT_limit_C=DELTA_T_LIMIT_C,
    heat_sampling='Q(t_k) held on [t_k,t_k+1); the terminal record is excluded.',
    initial_condition='Battery/coolant initial temperature = fixed inlet = ambient.',
    heat_temperature_feedback='Prescribed mission heat reused at 20/25/35 C; no electrothermal feedback.',
    solver='BTMS_model.solve_coolant_temperature_distribution; scipy least_squares trf; dt=1 s.',
    solver_settings=SOLVER_SETTINGS,
    balance_residual_tol_W=BALANCE_RESIDUAL_TOL_W,
    energy_closure_tol_J=ENERGY_CLOSURE_TOL_J,
    calculation_time_scope='New run_case solves only; reused cases have zero new solve time. Lookup/read/validation time is Result_load_time_s; Excel export excluded.',
    scope='AC plate/fan and LC cold plate/contained water/pump; radiator excluded.',
    refresh='Static simulation values. Re-run this notebook to update the workbook.',
)
parameter_rows = []
def append_parameters(prefix, value):
    if isinstance(value, dict):
        for key, item in value.items():
            append_parameters(f'{prefix}.{key}' if prefix else key, item)
    else:
        parameter_rows.append({'Parameter': prefix, 'Value': value})

append_parameters('', configuration)
for row in source_table.to_dict(orient='records'):
    mission_key = row.pop('Mission_key')
    append_parameters(f'heat_source.{mission_key}', row)
parameter_table = pd.DataFrame(parameter_rows)

step_checks = verification_table.copy()
if not step_checks.empty:
    step_checks.insert(0, 'Check', 'Original solver step')
    step_checks['Status'] = 'checked'
baseline_checks = regression_table.copy()
baseline_checks.insert(0, 'Check', 'Baseline time history')
validation_table = pd.concat([step_checks, baseline_checks], ignore_index=True, sort=False)
validation_columns = ['Check', 'Cooling', 'Mission_key', 'Ambient_C', 'time_s',
                      'Variable', 'max_abs_difference_C', 'Status', 'Reference']
validation_table = validation_table.reindex(columns=validation_columns)


def write_excel_tables(path, tables):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix('.tmp.xlsx')
    try:
        with pd.ExcelWriter(temporary_path, engine='openpyxl') as writer:
            for sheet_name, table in tables.items():
                table.to_excel(writer, sheet_name=sheet_name, index=False)
                sheet = writer.sheets[sheet_name]
                sheet.freeze_panes = 'A2'
                sheet.auto_filter.ref = sheet.dimensions
                sheet.sheet_view.showGridLines = False
                sheet.row_dimensions[1].height = 36
                for cell in sheet[1]:
                    cell.font = Font(name='Arial', size=10, bold=True, color='FFFFFF')
                    cell.fill = PatternFill('solid', fgColor='26475F')
                    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
                for column_index, column in enumerate(table.columns, start=1):
                    width = 22
                    if column in ('time', 'time_s', 'start_s', 'end_s'):
                        width = 12
                    elif column in ('CaseID', 'case_id'):
                        width = 26
                    elif column in ('Reference', 'Heat_source', 'source_notebook', 'Workbook', 'Result_source'):
                        width = max(30, max((len(str(v)) for v in table[column].dropna()), default=20)+2)
                    elif column.endswith('sha256') or column == 'Input_signature':
                        width = 68
                    elif column == 'Reuse_note':
                        width = 85
                    elif column == 'Parameter':
                        width = 48
                    elif column == 'Value':
                        width = 110
                    sheet.column_dimensions[get_column_letter(column_index)].width = width
                    if pd.api.types.is_float_dtype(table[column]):
                        number_format = '0.000'
                        if any(token in column for token in ('residual', 'closure', 'difference_C')):
                            number_format = '0.000E+00'
                        if column in ('time', 'time_s', 'interval_end_s'):
                            number_format = '0'
                        for cells in sheet.iter_cols(min_col=column_index, max_col=column_index, min_row=2):
                            for cell in cells:
                                cell.number_format = number_format
                if 'Reuse_note' in table:
                    note_index = table.columns.get_loc('Reuse_note') + 1
                    for cells in sheet.iter_cols(min_col=note_index, max_col=note_index, min_row=2):
                        for cell in cells:
                            cell.alignment = Alignment(wrap_text=True, vertical='center')
                            sheet.row_dimensions[cell.row].height = 36
                if sheet_name == 'Parameters':
                    for cells in sheet.iter_rows(min_row=2, min_col=2, max_col=2):
                        cells[0].alignment = Alignment(wrap_text=True, vertical='center')
                        sheet.row_dimensions[cells[0].row].height = 30
        os.replace(temporary_path, path)
    finally:
        if temporary_path.exists():
            temporary_path.unlink()


def build_case_tables(case_id):
    history, detailed = histories[case_id], temperatures[case_id]
    base = summary_table.loc[summary_table['CaseID'].eq(case_id)].iloc[0].to_dict()
    cooling, mission_key, ambient = base['Cooling'], base['Mission_key'], base['Ambient_C']
    mission, d = MISSIONS[mission_key], designs[cooling, ambient]
    outlet_name = 'Twater_out_C' if cooling == 'LC' else 'Tair_out_C'
    medium_prefix = 'T_water' if cooling == 'LC' else 'T_air'
    # Match C0009's first 45 Selected columns (air uses the corresponding air names).
    selected = pd.DataFrame({
        'time_s': history['time_s'], 'Tmax_C': history['Tmax_C'],
        'Tmin_C': history['Tmin_C'], 'DeltaT_C': history['DeltaT_C'],
        outlet_name: history['Tcool_out_C'],
    })
    for i in range(d['segments']):
        selected[f'T_bat_seg_{i+1:02d}_C'] = detailed[f'T_bat_{i+1:02d}_C']
    for i in range(d['segments']):
        selected[f'{medium_prefix}_seg_{i+1:02d}_C'] = detailed[f'T_cool_{i+1:02d}_C']
    for column in ('Tmean_C', 'Stage', 'Q_cell_next_interval_W', 'P_aux_next_interval_W',
                   'E_aux_cumulative_J', 'E_heat_module_cumulative_J',
                   'E_removed_module_cumulative_J', 'E_stored_module_J'):
        selected[column] = history[column]

    # Exact C0009 Sheet1 names and order; only one continuous case/time axis.
    nodes = pd.DataFrame({'time': detailed['time_s']})
    for i in range(d['segments']):
        nodes[f'T_bat_{i+1}'] = detailed[f'T_bat_{i+1:02d}_C']
    for i in range(d['segments']):
        nodes[f'T_cool_{i+1}'] = detailed[f'T_cool_{i+1:02d}_C']
    if cooling == 'LC':
        dh = 2*LC['width_m']*LC['height_m']/(LC['width_m']+LC['height_m'])
        mass = BTMS_model.cal_btms_mass(dict(
            fluid_cool='Water', rho_cool=d['props']['rho'], rho_plate=LC['rho_plate_kg_m3'],
            L_channel=LC['length_m'], W_plate=d['width_m'], A_cool_cs=d['area_cs'],
            H_channel=LC['height_m'], H_bottom=LC['bottom_m'], num_channel=LC['channels'],
            m_pump=0.0, m_pipe=0.0, return_components=True))
        row = dict(
            model='Selected', case_id=case_id, T_water_in_C=ambient,
            m_dot_total_kg_s=LC['m_dot_total_kg_s'],
            m_dot_channel_kg_s=LC['m_dot_total_kg_s']/LC['channels'],
            num_cell_seg=d['cells_per_node'], num_parallel_channels=LC['channels'],
            W_channel_mm=LC['width_m']*1000, H_channel_mm=LC['height_m']*1000,
            L_channel_mm=LC['length_m']*1000, W_plate_mm=d['width_m']*1000,
            S_channel_mm=(d['width_m']-LC['channels']*LC['width_m'])/(LC['channels']+1)*1000,
            spacing_mode='auto', H_plate_mm=(LC['height_m']+LC['bottom_m'])*1000,
            Dh_mm=dh*1000, u_water_m_s=d['velocity'], Re_water=d['Re'], Nu_water=d['Nu'],
            h_water_W_m2K=d['Nu']*d['props']['k']/dh, htc_global_W_m2K=d['htc'],
            P_pump=d['pump_power_W'], E_pump_cumulative=base['E_aux_J'],
            m_plate_kg=mass['m_plate_kg'], m_coolant_kg=mass['m_coolant_kg'],
            mtotal=mass['m_BTMS_kg'], Tmax_mission_C=base['Tmax_C'],
            DeltaT_mission_max_C=base['DeltaTmax_C'],
            Twater_out_cruise_end_C=float(history.loc[mission['bounds_s'][4], 'Tcool_out_C']),
        )
        for i, stage in enumerate(STAGE_NAMES):
            start, end = mission['bounds_s'][i:i+2].astype(int)
            row[f'dTmax_{stage}_C'] = float(history.loc[end,'Tmax_C']-history.loc[start,'Tmax_C'])
        row.update(source_notebook='C0010_PARK25_ACLC_INTG_1D_EVAL.ipynb',
                   simulation_duration_s=mission['duration_s'], dt_s=DT_S)
    else:
        st = BATTERY['diameter_m']+AC['gap_m']
        row = dict(
            model='Selected', case_id=case_id, V_air=AC['velocity_m_s'], T_air=ambient,
            s_gap_mm=AC['gap_m']*1000, cooling_mode='all_mission',
            source_notebook='C0010_PARK25_ACLC_INTG_1D_EVAL.ipynb',
            E_fan_cumulative_J=base['E_aux_J'], E_fan_cumulative_kJ=base['E_aux_J']/1000,
            Re_air=d['Re'], Nu_air=d['Nu'], h_air_W_m2K=d['htc'],
            S_T_mm=st*1000, S_D_mm=st*1000, S_L_mm=np.sqrt(3)/2*st*1000,
            L_module_m=d['length_m'], W_module_m=d['width_m'], m_bottom_plate_kg=d['mass_kg'],
            mission_Tmax_C=base['Tmax_C'], mission_DeltaT_max_C=base['DeltaTmax_C'],
            time_of_Tmax_s=base['time_of_Tmax_s'], final_Tmin_C=base['final_Tmin_C'],
            simulation_duration_s=mission['duration_s'], dt_s=DT_S)
    base['model'] = 'Selected'
    row.update(base)
    row.update(cruise_s=mission['cruise_s'], hover_s=mission['hover_s'])
    for i, stage in enumerate(STAGE_NAMES):
        row[f'stage_{stage}_start_s'] = int(mission['bounds_s'][i])
        row[f'stage_{stage}_end_s'] = int(mission['bounds_s'][i+1])
    return {'Summary': pd.DataFrame([row]), 'Selected': selected, 'Sheet1': nodes}


case_workbook_tables = {}
workbook_paths = {}
for case_id in histories:
    tables = build_case_tables(case_id)
    duration = int(tables['Summary']['simulation_duration_s'].iloc[0])
    if not np.array_equal(tables['Selected']['time_s'], np.arange(duration+1)):
        raise ValueError(f'Incomplete mission time axis: {case_id}')
    if not np.array_equal(tables['Selected']['time_s'], tables['Sheet1']['time']):
        raise ValueError(f'Selected and Sheet1 time axes differ: {case_id}')
    workbook_path = CASE_WORKBOOK_DIR / f'{RESULT_BASENAME}_{case_id}.xlsx'
    write_excel_tables(workbook_path, tables)
    case_workbook_tables[case_id] = tables
    workbook_paths[case_id] = workbook_path.relative_to(RESULT_EXCEL_PATH.parent).as_posix()
    print(f'Saved {case_id}: {duration+1} full-mission states; Summary / Selected / Sheet1')

index_summary = summary_table.copy()
index_summary['model'] = 'Selected'
index_summary['cruise_s'] = index_summary['Mission_key'].map(lambda key: MISSIONS[key]['cruise_s'])
index_summary['hover_s'] = index_summary['Mission_key'].map(lambda key: MISSIONS[key]['hover_s'])
index_summary['Workbook'] = index_summary['CaseID'].map(workbook_paths)
write_excel_tables(RESULT_EXCEL_PATH, {
    'Summary': index_summary, 'Stage_summary': stage_table,
    'Heat_generation': heat_generation_table, 'Parameters': parameter_table,
    'Validation': validation_table, 'Run_status': reuse_table,
})
print('Saved comparison index:', RESULT_EXCEL_PATH)
print('Saved case workbooks:', CASE_WORKBOOK_DIR)
print('Output: 18 complete case workbooks and one comparison index; Excel files only.')


Saved AC_short_T020C: 1294 full-mission states; Summary / Selected / Sheet1
Saved LC_short_T020C: 1294 full-mission states; Summary / Selected / Sheet1
Saved AC_short_T025C: 1294 full-mission states; Summary / Selected / Sheet1
Saved LC_short_T025C: 1294 full-mission states; Summary / Selected / Sheet1
Saved AC_short_T035C: 1294 full-mission states; Summary / Selected / Sheet1
Saved LC_short_T035C: 1294 full-mission states; Summary / Selected / Sheet1
Saved AC_baseline_T020C: 1921 full-mission states; Summary / Selected / Sheet1
Saved LC_baseline_T020C: 1921 full-mission states; Summary / Selected / Sheet1
Saved AC_baseline_T025C: 1921 full-mission states; Summary / Selected / Sheet1
Saved LC_baseline_T025C: 1921 full-mission states; Summary / Selected / Sheet1
Saved AC_baseline_T035C: 1921 full-mission states; Summary / Selected / Sheet1
Saved LC_baseline_T035C: 1921 full-mission states; Summary / Selected / Sheet1
Saved AC_long_T020C: 2221 full-mission states; Summary / Selected / Sh